# Master Training Pipeline: Optuna Hybrid
**Architecture:** Tri-Layer Hybrid
1. Isolation Forest + IQR (Anomaly Detection & Imputation)
2. Prophet (Base Macroscopic Trending)
3. LightGBM (Micro Residual Corrections)

All governed by Joint Bayesian Optimization (Optuna).

**Key Design:** Anomalous values are *imputed* with the trailing 7-day mean of clean data, so no rows are dropped and the time series remains gap-free.

## 1. Setup & Data Ingestion

In [ ]:
import os
import json
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import lightgbm as lgb
import optuna
import shap
import joblib
from prophet import Prophet
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    precision_score,
    recall_score,
    f1_score,
)

matplotlib.use('Agg')
warnings.filterwarnings('ignore')

# Silence cmdstanpy logs from Prophet
cmdstanpy_logger = logging.getLogger('cmdstanpy')
cmdstanpy_logger.addHandler(logging.NullHandler())
cmdstanpy_logger.propagate = False
cmdstanpy_logger.setLevel(logging.CRITICAL)

In [ ]:
# --- Path Configuration ---
OUTPUT_DIR = '../Outputs'
MODELS_DIR = '../Models'
os.makedirs(MODELS_DIR, exist_ok=True)

PARAMS_PATH = os.path.join(MODELS_DIR, 'best_hybrid_params.json')

# Tuning control
OPTUNA_TRIALS = 30
RETUNE_EVERY_DAYS = 30
FORCE_RETUNE = False

In [ ]:
# --- Load & Interpolate ---
print('1. Pre-processing: Raw data ingestion...')
df = pd.read_csv(os.path.join(OUTPUT_DIR, 'dataset_daily_processed.csv'))
df['Date'] = pd.to_datetime(df['Date'])

print('2. Handling missing values via temporal interpolation...')
df.set_index('Date', inplace=True)
df = df.interpolate(method='time')
df.reset_index(inplace=True)

## 2. Feature Engineering & Chronological Split

In [ ]:
print('3. Feature engineering (lag variables, rolling average, calendar features)...')

TARGET_COL = 'Demand_MWh'
FEATURE_CANDIDATES = [
    'Day_of_Week', 'Is_Weekend', 'Is_Holiday',
    'Avg_Temp', 'Rainfall',
    'Lag_1', 'Lag_7', 'Lag_30', 'Rolling_7',
]
features = [c for c in FEATURE_CANDIDATES if c in df.columns]
df = df.dropna(subset=features + [TARGET_COL]).copy()
print(f'   Available features: {features}')

In [ ]:
print('4. Time-based split 70/15/15...')
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()
print(f'   Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

## 3. Hyperparameter Management (Save / Load / Retune)

In [ ]:
REQUIRED_PARAM_KEYS = {
    'contamination',
    'changepoint_prior_scale',
    'seasonality_prior_scale',
    'learning_rate',
    'max_depth',
    'num_leaves',
    'subsample',
    'colsample_bytree',
}


def load_saved_params(path: str) -> dict | None:
    """Load previously-tuned parameters from disk, with validation."""
    if not os.path.exists(path):
        return None
    try:
        with open(path, 'r', encoding='utf-8') as f:
            payload = json.load(f)
        params = payload.get('best_params', payload)
        if not isinstance(params, dict) or not REQUIRED_PARAM_KEYS.issubset(params.keys()):
            return None
        return payload
    except Exception as exc:
        print(f'   Warning: failed to read saved params ({exc}).')
        return None


def should_retune(saved_payload: dict | None) -> bool:
    """Decide whether to re-run Optuna based on age or flag."""
    if FORCE_RETUNE:
        print('   Retune reason: FORCE_RETUNE=True')
        return True
    if saved_payload is None:
        print('   Retune reason: no saved parameter file found.')
        return True
    tuned_at = saved_payload.get('last_tuned_at')
    if not tuned_at:
        print('   Retune reason: missing last_tuned_at metadata.')
        return True
    try:
        age_days = (pd.Timestamp.now() - pd.to_datetime(tuned_at)).days
        if age_days >= RETUNE_EVERY_DAYS:
            print(f'   Retune reason: params age {age_days} days >= {RETUNE_EVERY_DAYS} days.')
            return True
        print(f'   Using saved params (age: {age_days} days, retune threshold: {RETUNE_EVERY_DAYS} days).')
        return False
    except Exception:
        print('   Retune reason: invalid last_tuned_at format.')
        return True

## 4. Joint Bayesian Optimization (Optuna)

In [ ]:
print('5. Joint Bayesian Optimization (Optuna)...')
optuna.logging.set_verbosity(optuna.logging.WARNING)

df_prophet_val_proxy = val_df[['Date']].rename(columns={'Date': 'ds'})


def objective(trial: optuna.Trial) -> float:
    """Single Optuna trial: anomaly imputation -> Prophet -> LightGBM residual."""
    # --- Suggest joint parameters ---
    contamination = trial.suggest_float('contamination', 0.001, 0.05, log=True)
    cps = trial.suggest_float('changepoint_prior_scale', 0.01, 0.5, log=True)
    sps = trial.suggest_float('seasonality_prior_scale', 0.1, 10.0, log=True)
    lgb_lr = trial.suggest_float('learning_rate', 0.01, 0.1, log=True)
    lgb_depth = trial.suggest_int('max_depth', 3, 7)
    lgb_leaves = trial.suggest_int('num_leaves', 15, 63)
    lgb_subsample = trial.suggest_float('subsample', 0.7, 1.0)
    lgb_colsample = trial.suggest_float('colsample_bytree', 0.7, 1.0)

    # --- Anomaly Detection + Imputation (IQR + Isolation Forest) ---
    temp_forest = IsolationForest(
        n_estimators=100, max_samples='auto',
        contamination=contamination, random_state=42, n_jobs=-1,
    )
    temp_forest.fit(train_df[features])
    if_anomalies = temp_forest.predict(train_df[features])

    q1 = train_df[TARGET_COL].quantile(0.25)
    q3 = train_df[TARGET_COL].quantile(0.75)
    iqr = q3 - q1
    iqr_anomalies = np.where(
        (train_df[TARGET_COL] < q1 - 1.5 * iqr) | (train_df[TARGET_COL] > q3 + 1.5 * iqr),
        -1, 1,
    )
    is_anomaly = (if_anomalies == -1) | (iqr_anomalies == -1)

    temp_train = train_df.copy()
    for idx in np.where(is_anomaly)[0]:
        start = max(0, idx - 7)
        clean_mask = ~is_anomaly[start:idx]
        clean_window = train_df[TARGET_COL].iloc[start:idx][clean_mask]
        imputed = clean_window.mean() if len(clean_window) > 0 else train_df[TARGET_COL].mean()
        temp_train.iloc[idx, temp_train.columns.get_loc(TARGET_COL)] = imputed

    # --- Prophet baseline ---
    df_p = temp_train[['Date', TARGET_COL]].rename(columns={'Date': 'ds', TARGET_COL: 'y'})
    m = Prophet(
        yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False,
        changepoint_prior_scale=cps, seasonality_prior_scale=sps,
    )
    m.fit(df_p)

    temp_train['Prophet_Pred'] = m.predict(df_p)['yhat'].values
    temp_val = val_df.copy()
    temp_val['Prophet_Pred'] = m.predict(df_prophet_val_proxy)['yhat'].values

    # --- LightGBM residual correction ---
    temp_train['Prophet_Residual'] = temp_train[TARGET_COL] - temp_train['Prophet_Pred']
    temp_val['Prophet_Residual'] = temp_val[TARGET_COL] - temp_val['Prophet_Pred']

    lgb_model = lgb.LGBMRegressor(
        learning_rate=lgb_lr, max_depth=lgb_depth, num_leaves=lgb_leaves,
        subsample=lgb_subsample, colsample_bytree=lgb_colsample,
        n_estimators=1000, random_state=42, n_jobs=-1, verbose=-1,
    )
    lgb_model.fit(
        temp_train[features], temp_train['Prophet_Residual'],
        eval_set=[(temp_val[features], temp_val['Prophet_Residual'])],
        callbacks=[
            lgb.early_stopping(stopping_rounds=20, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )

    hybrid_preds = temp_val['Prophet_Pred'] + lgb_model.predict(temp_val[features])
    return mean_absolute_error(temp_val[TARGET_COL], hybrid_preds)

In [ ]:
saved_payload = load_saved_params(PARAMS_PATH)

if should_retune(saved_payload):
    print(f'   Running Joint Bayesian Search ({OPTUNA_TRIALS} Trials)...')
    sampler = optuna.samplers.TPESampler(seed=0)
    study = optuna.create_study(direction='minimize', sampler=sampler)
    study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

    best_p = study.best_params
    print(f'   Best Joint Parameters: {best_p}')

    payload = {
        'best_params': best_p,
        'last_tuned_at': pd.Timestamp.now().isoformat(),
        'n_trials': OPTUNA_TRIALS,
        'retune_every_days': RETUNE_EVERY_DAYS,
        'target_col': TARGET_COL,
        'features': features,
    }
    with open(PARAMS_PATH, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2)
    print(f'   Saved best parameters to: {PARAMS_PATH}')
else:
    best_p = saved_payload['best_params']
    print(f'   Loaded saved best parameters from: {PARAMS_PATH}')

## 5. Final Architecture Training

In [ ]:
print('6. Training Final Champion Architecture...')

# --- 5a. Anomaly Detection + Imputation (IQR + Isolation Forest) ---
iso_forest = IsolationForest(
    n_estimators=300, max_samples='auto',
    contamination=best_p['contamination'], random_state=42, n_jobs=-1,
)
iso_forest.fit(train_df[features])
train_anomalies = iso_forest.predict(train_df[features])

q1 = train_df[TARGET_COL].quantile(0.25)
q3 = train_df[TARGET_COL].quantile(0.75)
iqr_val = q3 - q1
lower_bound = q1 - 1.5 * iqr_val
upper_bound = q3 + 1.5 * iqr_val
iqr_anomalies = np.where(
    (train_df[TARGET_COL] < lower_bound) | (train_df[TARGET_COL] > upper_bound),
    -1, 1,
)

is_anomaly = (train_anomalies == -1) | (iqr_anomalies == -1)
num_anomalies = is_anomaly.sum()

# Impute anomalous points with trailing 7-day clean mean
train_df_clean = train_df.copy()
for idx in np.where(is_anomaly)[0]:
    start = max(0, idx - 7)
    clean_mask = ~is_anomaly[start:idx]
    clean_window = train_df[TARGET_COL].iloc[start:idx][clean_mask]
    imputed = clean_window.mean() if len(clean_window) > 0 else train_df[TARGET_COL].mean()
    train_df_clean.iloc[idx, train_df_clean.columns.get_loc(TARGET_COL)] = imputed

print(f'   Imputed {num_anomalies} anomalies (0 rows removed).')

In [ ]:
# --- Isolation Forest vs IQR evaluation ---
y_true_anom = (iqr_anomalies == -1).astype(int)
y_pred_anom = (train_anomalies == -1).astype(int)

iso_precision = precision_score(y_true_anom, y_pred_anom, zero_division=0)
iso_recall = recall_score(y_true_anom, y_pred_anom, zero_division=0)
iso_f1 = f1_score(y_true_anom, y_pred_anom, zero_division=0)

print(f'   IF vs IQR — Precision: {iso_precision:.3f} | Recall: {iso_recall:.3f} | F1: {iso_f1:.3f}')

In [ ]:
# --- Anomaly Visualization ---
anomalous_data = train_df[train_anomalies == -1]

plt.figure(figsize=(15, 6))
plt.plot(train_df['Date'], train_df[TARGET_COL], color='royalblue', label='Normal Demand', alpha=0.6, linewidth=1)
plt.scatter(anomalous_data['Date'], anomalous_data[TARGET_COL], color='crimson', label='Detected Anomaly', zorder=5)
plt.title('Isolation Forest: Detected Anomalies in Training Data')
plt.xlabel('Date')
plt.ylabel('Electricity Demand (MWh)')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig0_anomalies_detected.png'), dpi=300)
plt.show()

In [ ]:
# --- 5b. Final Prophet Training ---
df_prophet_train = train_df_clean[['Date', TARGET_COL]].rename(columns={'Date': 'ds', TARGET_COL: 'y'})
prophet_model = Prophet(
    yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False,
    changepoint_prior_scale=best_p['changepoint_prior_scale'],
    seasonality_prior_scale=best_p['seasonality_prior_scale'],
)
prophet_model.fit(df_prophet_train)

for split_df in [train_df_clean, train_df, val_df, test_df]:
    future = split_df[['Date']].rename(columns={'Date': 'ds'})
    split_df['Prophet_Pred'] = prophet_model.predict(future)['yhat'].values

print('   Prophet training complete.')

In [ ]:
# --- 5c. Final LightGBM Residual Training ---
for split_df in [train_df_clean, train_df, val_df, test_df]:
    split_df['Prophet_Residual'] = split_df[TARGET_COL] - split_df['Prophet_Pred']

model_lgb = lgb.LGBMRegressor(
    learning_rate=best_p['learning_rate'],
    max_depth=best_p['max_depth'],
    num_leaves=best_p['num_leaves'],
    subsample=best_p['subsample'],
    colsample_bytree=best_p['colsample_bytree'],
    n_estimators=5000, random_state=42, n_jobs=-1, verbose=-1,
)
model_lgb.fit(
    train_df_clean[features], train_df_clean['Prophet_Residual'],
    eval_set=[(val_df[features], val_df['Prophet_Residual'])],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=0),
    ],
)

for split_df in [train_df, val_df, test_df]:
    split_df['LGBM_Residual_Pred'] = model_lgb.predict(split_df[features])
    split_df['Final_Pred'] = split_df['Prophet_Pred'] + split_df['LGBM_Residual_Pred']

print('   LightGBM residual training complete.')

## 6. Comprehensive Model Evaluation

In [ ]:
def calc_mape(actual: np.ndarray, predicted: np.ndarray) -> float:
    """Mean Absolute Percentage Error — avoids division by zero."""
    mask = actual != 0
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100


# Prophet-Only
prophet_mae_val = mean_absolute_error(val_df[TARGET_COL], val_df['Prophet_Pred'])
prophet_rmse_val = np.sqrt(mean_squared_error(val_df[TARGET_COL], val_df['Prophet_Pred']))
prophet_mape_val = calc_mape(val_df[TARGET_COL].values, val_df['Prophet_Pred'].values)

prophet_mae_test = mean_absolute_error(test_df[TARGET_COL], test_df['Prophet_Pred'])
prophet_rmse_test = np.sqrt(mean_squared_error(test_df[TARGET_COL], test_df['Prophet_Pred']))
prophet_mape_test = calc_mape(test_df[TARGET_COL].values, test_df['Prophet_Pred'].values)

# Hybrid
hybrid_mae_val = mean_absolute_error(val_df[TARGET_COL], val_df['Final_Pred'])
hybrid_rmse_val = np.sqrt(mean_squared_error(val_df[TARGET_COL], val_df['Final_Pred']))
hybrid_mape_val = calc_mape(val_df[TARGET_COL].values, val_df['Final_Pred'].values)

hybrid_mae_test = mean_absolute_error(test_df[TARGET_COL], test_df['Final_Pred'])
hybrid_rmse_test = np.sqrt(mean_squared_error(test_df[TARGET_COL], test_df['Final_Pred']))
hybrid_mape_test = calc_mape(test_df[TARGET_COL].values, test_df['Final_Pred'].values)

# Display
print('\n' + '=' * 72)
print('  MODEL COMPARISON: Prophet-Only vs Hybrid (Prophet + LightGBM)')
print('=' * 72)
print(f"{'Metric':<12} | {'Prophet-Only (Val)':<22} | {'Hybrid (Val)':<22}")
print('-' * 72)
print(f"{'MAE':<12} | {prophet_mae_val:>18,.2f} MWh | {hybrid_mae_val:>18,.2f} MWh")
print(f"{'RMSE':<12} | {prophet_rmse_val:>18,.2f} MWh | {hybrid_rmse_val:>18,.2f} MWh")
print(f"{'MAPE':<12} | {prophet_mape_val:>17.2f}%     | {hybrid_mape_val:>17.2f}%")
print('-' * 72)
print(f"{'Metric':<12} | {'Prophet-Only (Test)':<22} | {'Hybrid (Test)':<22}")
print('-' * 72)
print(f"{'MAE':<12} | {prophet_mae_test:>18,.2f} MWh | {hybrid_mae_test:>18,.2f} MWh")
print(f"{'RMSE':<12} | {prophet_rmse_test:>18,.2f} MWh | {hybrid_rmse_test:>18,.2f} MWh")
print(f"{'MAPE':<12} | {prophet_mape_test:>17.2f}%     | {hybrid_mape_test:>17.2f}%")
print('=' * 72)

rmse_improvement = ((prophet_rmse_test - hybrid_rmse_test) / prophet_rmse_test) * 100
mape_improvement = ((prophet_mape_test - hybrid_mape_test) / prophet_mape_test) * 100
print(f'\n  >> Hybrid improves Test RMSE by {rmse_improvement:.1f}%')
print(f'  >> Hybrid improves Test MAPE by {mape_improvement:.1f}%')

## 7. Export Models & Predictions

In [ ]:
# Serialize models
joblib.dump(prophet_model, os.path.join(MODELS_DIR, 'prophet_model.joblib'))
joblib.dump(model_lgb, os.path.join(MODELS_DIR, 'lgbm_model.joblib'))
joblib.dump(iso_forest, os.path.join(MODELS_DIR, 'iso_forest.joblib'))
print(f'   Models saved to: {MODELS_DIR}')

# Export predictions CSV for the dashboard
df_all = pd.concat([train_df, val_df, test_df]).sort_values('Date').reset_index(drop=True)
df_all.rename(columns={'Final_Pred': 'Hybrid_Prediction'}, inplace=True)
predictions_path = os.path.join(OUTPUT_DIR, 'dataset_daily_with_predictions.csv')
df_all.to_csv(predictions_path, index=False)
print(f'   Predictions saved to: {predictions_path}')

In [ ]:
# XAI SHAP export
def export_xai_plot(lgbm_model: lgb.LGBMRegressor, x_data: pd.DataFrame, output_path: str) -> None:
    """Export a SHAP feature-impact visualization, with LightGBM importance as fallback."""
    sample_n = min(500, len(x_data))
    x_sample = x_data.sample(n=sample_n, random_state=42) if len(x_data) > sample_n else x_data
    try:
        explainer = shap.TreeExplainer(lgbm_model)
        shap_vals = explainer.shap_values(x_sample)
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_vals, x_sample, show=False)
        plt.title('Global SHAP Summary for Exogenous Features')
    except Exception as exc:
        print(f'   Warning: SHAP export failed ({exc}), using fallback.')
        importances = pd.Series(lgbm_model.feature_importances_, index=x_data.columns).sort_values(ascending=True)
        plt.figure(figsize=(10, 6))
        importances.plot(kind='barh')
        plt.title('Feature Importance (Fallback)')
        plt.xlabel('Importance')
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()
    print(f'   XAI plot saved to: {output_path}')


export_xai_plot(model_lgb, df[features], os.path.join(OUTPUT_DIR, 'output_xai.png'))

## 8. Visualizations

In [ ]:
# Professional dark theme
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d29',
    'axes.edgecolor': '#2d3250',
    'axes.labelcolor': '#e0e0e0',
    'text.color': '#e0e0e0',
    'xtick.color': '#a0a0a0',
    'ytick.color': '#a0a0a0',
    'grid.color': '#2d3250',
    'grid.alpha': 0.5,
    'font.family': 'sans-serif',
    'font.size': 10,
})

In [ ]:
# --- Figure 1: Actual vs Predicted (Full Timeline) ---
all_dates = pd.concat([train_df['Date'], val_df['Date'], test_df['Date']])
all_actual = pd.concat([train_df[TARGET_COL], val_df[TARGET_COL], test_df[TARGET_COL]])
all_prophet = pd.concat([train_df['Prophet_Pred'], val_df['Prophet_Pred'], test_df['Prophet_Pred']])
all_hybrid = pd.concat([
    train_df.get('Hybrid_Prediction', train_df['Final_Pred'] if 'Final_Pred' in train_df.columns else train_df['Prophet_Pred']),
    val_df.get('Hybrid_Prediction', val_df['Final_Pred'] if 'Final_Pred' in val_df.columns else val_df['Prophet_Pred']),
    test_df.get('Hybrid_Prediction', test_df['Final_Pred'] if 'Final_Pred' in test_df.columns else test_df['Prophet_Pred']),
])

fig1, ax1 = plt.subplots(figsize=(16, 6))
ax1.plot(all_dates, all_actual, color='#4fc3f7', alpha=0.6, linewidth=0.7, label='Actual Demand')
ax1.plot(all_dates, all_prophet, color='#ff8a65', linewidth=0.9, linestyle='--', alpha=0.7, label='Prophet-Only')
ax1.plot(all_dates, all_hybrid, color='#66bb6a', linewidth=1.0, alpha=0.85, label='Hybrid (Prophet+LGB)')
ax1.axvspan(train_df['Date'].iloc[0], train_df['Date'].iloc[-1], alpha=0.04, color='#4fc3f7', label='Train (70%)')
ax1.axvspan(val_df['Date'].iloc[0], val_df['Date'].iloc[-1], alpha=0.08, color='#ffab40', label='Validation (15%)')
ax1.axvspan(test_df['Date'].iloc[0], test_df['Date'].iloc[-1], alpha=0.08, color='#ef5350', label='Test (15%)')
ax1.set_title('Electricity Demand: Actual vs Model Predictions (Full Timeline)', fontsize=14, fontweight='bold', pad=15)
ax1.set_xlabel('Date')
ax1.set_ylabel('Demand (MWh)')
ax1.legend(loc='upper left', fontsize=8, ncol=3, framealpha=0.3)
ax1.grid(True, alpha=0.3)
fig1.tight_layout()
fig1.savefig(os.path.join(OUTPUT_DIR, 'fig1_actual_vs_predicted.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Figure 2: Model Comparison Bar Chart (MAE, RMSE, MAPE) ---
models_list = ['Prophet-Only', 'Hybrid\n(Prophet+LGB)']
colors_bar = ['#ff8a65', '#66bb6a']
mae_pct = ((prophet_mae_test - hybrid_mae_test) / prophet_mae_test) * 100

fig2, (ax2a, ax2b, ax2c) = plt.subplots(1, 3, figsize=(18, 5))

# MAE
mae_vals = [prophet_mae_test, hybrid_mae_test]
bars0 = ax2a.bar(models_list, mae_vals, color=colors_bar, width=0.5, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars0, mae_vals):
    ax2a.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 200, f'{val:,.0f}',
              ha='center', va='bottom', fontweight='bold', fontsize=11, color='#e0e0e0')
ax2a.set_title('Test Set MAE', fontsize=13, fontweight='bold', pad=12)
ax2a.set_ylabel('MAE (MWh)')
ax2a.grid(axis='y', alpha=0.3)
if hybrid_mae_test < prophet_mae_test:
    ax2a.annotate(f'{mae_pct:.1f}% better', xy=(1, hybrid_mae_test), fontsize=10,
                  color='#66bb6a', ha='center', va='top', fontweight='bold')

# RMSE
rmse_vals = [prophet_rmse_test, hybrid_rmse_test]
bars1 = ax2b.bar(models_list, rmse_vals, color=colors_bar, width=0.5, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars1, rmse_vals):
    ax2b.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 200, f'{val:,.0f}',
              ha='center', va='bottom', fontweight='bold', fontsize=11, color='#e0e0e0')
ax2b.set_title('Test Set RMSE', fontsize=13, fontweight='bold', pad=12)
ax2b.set_ylabel('RMSE (MWh)')
ax2b.grid(axis='y', alpha=0.3)
if hybrid_rmse_test < prophet_rmse_test:
    ax2b.annotate(f'{rmse_improvement:.1f}% better', xy=(1, hybrid_rmse_test), fontsize=10,
                  color='#66bb6a', ha='center', va='top', fontweight='bold')

# MAPE
mape_vals = [prophet_mape_test, hybrid_mape_test]
bars2 = ax2c.bar(models_list, mape_vals, color=colors_bar, width=0.5, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars2, mape_vals):
    ax2c.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1, f'{val:.2f}%',
              ha='center', va='bottom', fontweight='bold', fontsize=11, color='#e0e0e0')
ax2c.set_title('Test Set MAPE', fontsize=13, fontweight='bold', pad=12)
ax2c.set_ylabel('MAPE (%)')
ax2c.grid(axis='y', alpha=0.3)
if hybrid_mape_test < prophet_mape_test:
    ax2c.annotate(f'{mape_improvement:.1f}% better', xy=(1, hybrid_mape_test), fontsize=10,
                  color='#66bb6a', ha='center', va='top', fontweight='bold')

fig2.suptitle('Model Accuracy Comparison: Prophet-Only vs Hybrid', fontsize=15, fontweight='bold', y=1.02, color='#ffffff')
fig2.tight_layout()
fig2.savefig(os.path.join(OUTPUT_DIR, 'fig2_model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Figure 3: Residual Distribution (Prophet vs Hybrid) ---
hybrid_col = 'Hybrid_Prediction' if 'Hybrid_Prediction' in test_df.columns else 'Final_Pred'
prophet_residuals_test = test_df[TARGET_COL] - test_df['Prophet_Pred']
hybrid_residuals_test = test_df[TARGET_COL] - test_df[hybrid_col]

fig3, (ax3a, ax3b) = plt.subplots(1, 2, figsize=(14, 5))

ax3a.hist(prophet_residuals_test, bins=40, color='#ff8a65', alpha=0.8, edgecolor='#1a1d29')
ax3a.axvline(x=0, color='white', linestyle='--', linewidth=1, alpha=0.7)
ax3a.set_title('Prophet-Only Residuals (Test)', fontsize=12, fontweight='bold')
ax3a.set_xlabel('Residual (MWh)')
ax3a.set_ylabel('Frequency')
ax3a.grid(axis='y', alpha=0.3)

ax3b.hist(hybrid_residuals_test, bins=40, color='#66bb6a', alpha=0.8, edgecolor='#1a1d29')
ax3b.axvline(x=0, color='white', linestyle='--', linewidth=1, alpha=0.7)
ax3b.set_title('Hybrid Residuals (Test)', fontsize=12, fontweight='bold')
ax3b.set_xlabel('Residual (MWh)')
ax3b.set_ylabel('Frequency')
ax3b.grid(axis='y', alpha=0.3)

max_abs = max(prophet_residuals_test.abs().max(), hybrid_residuals_test.abs().max()) * 1.1
ax3a.set_xlim(-max_abs, max_abs)
ax3b.set_xlim(-max_abs, max_abs)

fig3.suptitle('Residual Distribution — Tighter = More Accurate', fontsize=14, fontweight='bold', y=1.02, color='#ffffff')
fig3.tight_layout()
fig3.savefig(os.path.join(OUTPUT_DIR, 'fig3_residual_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Figure 4: SHAP Feature Importance ---
explainer = shap.TreeExplainer(model_lgb)
shap_values = explainer.shap_values(test_df[features])
shap.summary_plot(shap_values, test_df[features], feature_names=features, show=False)

fig4 = plt.gcf()
fig4.set_facecolor('#0f1117')
fig4.set_size_inches(10, 6)
fig4.savefig(os.path.join(OUTPUT_DIR, 'fig4_shap_summary.png'), dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

In [ ]:
print('\n' + '=' * 72)
print('  ALL OUTPUTS SAVED:')
print(f'    Models      -> {MODELS_DIR}')
print(f'    Figures     -> {OUTPUT_DIR}')
print('    - fig0_anomalies_detected.png')
print('    - fig1_actual_vs_predicted.png')
print('    - fig2_model_comparison.png')
print('    - fig3_residual_distribution.png')
print('    - fig4_shap_summary.png')
print(f'    Predictions -> {predictions_path}')
print('=' * 72)
print('\nSUCCESS! Hybrid Architecture Training Completed.')